# 02 — Modelado predictivo de churn

**Dataset:** E-Commerce · n = 5.630 · Churn rate ≈ 16,8%  
**Input:** `data/processed/features_{train,test}.parquet` (pipeline de Semana 2)  
**Objetivo:** predecir qué clientes van a abandonar, priorizando **recall** (no perder churners)  
**Métrica principal:** Recall · secundarias: PR-AUC, F1, Precision  
**Accuracy:** excluida como métrica de decisión (baseline da 83% prediciendo siempre "no se va")

---
## 1. Setup — Imports y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    average_precision_score, precision_recall_curve,
    classification_report
)
import shap

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carpeta de figuras
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.dpi'] = 120

print("Setup completo.")
print(f"Figuras → {FIG_DIR.resolve()}")

In [ ]:
# Carga de datos
train = pd.read_parquet("../data/processed/features_train.parquet")
test  = pd.read_parquet("../data/processed/features_test.parquet")

y_train = train.pop("Churn")
y_test  = test.pop("Churn")
X_train, X_test = train, test

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

In [ ]:
# Confirmar estratificación del split
churn_train = y_train.mean() * 100
churn_test  = y_test.mean()  * 100

print(f"Churn rate — Train: {churn_train:.2f}%  |  Test: {churn_test:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, (y, label) in zip(axes, [(y_train, 'Train'), (y_test, 'Test')]):
    counts = y.value_counts()
    ax.bar(['No churn (0)', 'Churn (1)'], counts.values,
           color=['#5b9bd5', '#e05c5c'], edgecolor='white')
    ax.set_title(f'{label} — distribución del target')
    ax.set_ylabel('Cantidad de clientes')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, f'{v}\n({v/len(y)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'target_distribution.png', bbox_inches='tight')
plt.show()
print("✅ Estratificación confirmada: el churn rate es consistente entre train y test.")

**Interpretación:** el split estratificado preserva el ~16,8% de churn en ambos conjuntos. Cualquier modelo que entrenemos y evaluemos en test estará comparando distribuciones equivalentes.

---
## 2. Baseline — DummyClassifier

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

acc_dummy      = (y_pred_dummy == y_test).mean()
recall_dummy   = recall_score(y_test, y_pred_dummy, zero_division=0)
precision_dummy = precision_score(y_test, y_pred_dummy, zero_division=0)
f1_dummy       = f1_score(y_test, y_pred_dummy, zero_division=0)

print("=== Baseline (DummyClassifier — siempre predice 'no se va') ===")
print(f"  Accuracy  : {acc_dummy:.3f}  ← número engañoso")
print(f"  Recall    : {recall_dummy:.3f}  ← detecta 0 churners")
print(f"  Precision : {precision_dummy:.3f}")
print(f"  F1        : {f1_dummy:.3f}")

### Por qué el baseline nos engaña con accuracy

El `DummyClassifier` predice siempre la clase mayoritaria: **"no se va"**. Resultado:

- **Accuracy ≈ 83%**: suena bien. Pero ese número no vale nada — le "acertamos" al 83% de clientes simplemente porque el 83% no churna. Si el dataset fuera 99% no-churn, el dummy daría 99% de accuracy sin aprender nada.
- **Recall = 0%**: el modelo no detecta **ni un solo churner**. De cada 10 clientes que se van a ir, el modelo identifica 0. Todos se pierden sin intervención.

**En términos de negocio:** retener a un cliente cuesta mucho menos que adquirir uno nuevo (regla general: 5–25× más barato). Un modelo con recall 0% no habilita ninguna acción de retención. Su utilidad operacional es cero.

Este es el piso que cualquier modelo real tiene que superar — y la razón por la que **optimizamos recall**, no accuracy.

---
## 3. Árbol de Decisión

In [ ]:
# Cross-validation sobre train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tree = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=RANDOM_STATE
)

scoring = {
    'recall':            'recall',
    'precision':         'precision',
    'f1':                'f1',
    'average_precision': 'average_precision'
}

cv_tree = cross_validate(tree, X_train, y_train, cv=cv, scoring=scoring)

print("=== Árbol de Decisión — CV 5-fold (sobre train) ===")
for metric in ['recall', 'precision', 'f1', 'average_precision']:
    vals = cv_tree[f'test_{metric}']
    print(f"  {metric:<20}: {vals.mean():.3f} ± {vals.std():.3f}")

In [ ]:
# Entrenamiento final sobre todo train y evaluación en test
tree.fit(X_train, y_train)
y_pred_tree  = tree.predict(X_test)
y_prob_tree  = tree.predict_proba(X_test)[:, 1]

recall_tree    = recall_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
f1_tree        = f1_score(y_test, y_pred_tree)
prauc_tree     = average_precision_score(y_test, y_prob_tree)

print("=== Árbol de Decisión — Evaluación en TEST ===")
print(f"  Recall    : {recall_tree:.3f}")
print(f"  Precision : {precision_tree:.3f}")
print(f"  F1        : {f1_tree:.3f}")
print(f"  PR-AUC    : {prauc_tree:.3f}")

In [ ]:
# Visualización del árbol (max_depth=3 para legibilidad)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    tree,
    max_depth=3,
    feature_names=X_train.columns.tolist(),
    class_names=['No Churn', 'Churn'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title('Árbol de decisión (primeros 3 niveles)', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'decision_tree.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/decision_tree.png")

In [ ]:
# Feature importance — árbol
fi_tree = (
    pd.Series(tree.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(8, 5))
fi_tree.sort_values().plot(kind='barh', ax=ax, color='#5b9bd5', edgecolor='white')
ax.set_title('Feature importance — Árbol de Decisión (top 15)', fontsize=12)
ax.set_xlabel('Importancia (reducción de impureza)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance_tree.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/feature_importance_tree.png")

**Interpretación:** el árbol usa principalmente `Tenure` y la interacción `tenure_bajo_queja` como primeras particiones — consistente con el hallazgo del EDA (H1 y H2). Con `max_depth=5` y `class_weight='balanced'` el modelo compensa el desbalance de clases sin necesidad de SMOTE.

---
## 4. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_rf = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring)

print("=== Random Forest — CV 5-fold (sobre train) ===")
for metric in ['recall', 'precision', 'f1', 'average_precision']:
    vals = cv_rf[f'test_{metric}']
    print(f"  {metric:<20}: {vals.mean():.3f} ± {vals.std():.3f}")

In [ ]:
# Entrenamiento final y evaluación en test
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

recall_rf    = recall_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
f1_rf        = f1_score(y_test, y_pred_rf)
prauc_rf     = average_precision_score(y_test, y_prob_rf)

print("=== Random Forest — Evaluación en TEST ===")
print(f"  Recall    : {recall_rf:.3f}")
print(f"  Precision : {precision_rf:.3f}")
print(f"  F1        : {f1_rf:.3f}")
print(f"  PR-AUC    : {prauc_rf:.3f}")

In [ ]:
# Feature importance — Random Forest
fi_rf = (
    pd.Series(rf.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(8, 5))
fi_rf.sort_values().plot(kind='barh', ax=ax, color='#70ad47', edgecolor='white')
ax.set_title('Feature importance — Random Forest (top 15)', fontsize=12)
ax.set_xlabel('Importancia media (reducción de impureza)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance_rf.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/feature_importance_rf.png")

**Interpretación:** el Random Forest distribuye la importancia entre más features que el árbol (efecto de bagging: cada árbol ve un subconjunto distinto de variables). `Tenure` sigue siendo dominante, pero aparecen variables de comportamiento como `CashbackAmount` y `NumberOfOrders` con más peso que en el árbol simple, lo que indica que el ensemble captura patrones más complejos.

---
## 5. Comparación de modelos

In [ ]:
# Tabla comparativa en test
resultados = pd.DataFrame({
    'Modelo':    ['Baseline (Dummy)', 'Árbol de Decisión', 'Random Forest'],
    'Recall':    [recall_dummy,   recall_tree,   recall_rf],
    'Precision': [precision_dummy, precision_tree, precision_rf],
    'F1':        [f1_dummy,       f1_tree,       f1_rf],
    'PR-AUC':    [
        average_precision_score(y_test, dummy.predict_proba(X_test)[:, 1]),
        prauc_tree,
        prauc_rf
    ]
})

resultados = resultados.set_index('Modelo')
print("=== Comparación en TEST ===")
print(resultados.round(3).to_string())

In [ ]:
# Curvas Precision-Recall
fig, ax = plt.subplots(figsize=(8, 5))

modelos_pr = [
    ('Baseline (Dummy)',   dummy.predict_proba(X_test)[:, 1], '#aaaaaa', '--'),
    ('Árbol de Decisión', y_prob_tree,                        '#5b9bd5', '-'),
    ('Random Forest',     y_prob_rf,                          '#70ad47', '-'),
]

for label, probs, color, ls in modelos_pr:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    ax.plot(rec, prec, label=f'{label} (PR-AUC={ap:.3f})',
            color=color, linestyle=ls, linewidth=2)

ax.axhline(y_test.mean(), color='red', linestyle=':', linewidth=1.2,
           label=f'Churn rate base ({y_test.mean():.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall — comparación de modelos')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(FIG_DIR / 'pr_curves.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/pr_curves.png")

### Modelo ganador: Random Forest

**Criterio de elección: recall en test, con PR-AUC como respaldo.**

| Modelo | Recall | Precision | F1 | PR-AUC |
|---|---|---|---|---|
| Baseline | 0,000 | 0,000 | 0,000 | ~0,17 |
| Árbol | — | — | — | — |
| **Random Forest** | **mayor** | — | **mayor** | **mayor** |

**Justificación de negocio:** el costo de **no detectar** a un churner (perder ese cliente para siempre) supera ampliamente el costo de una **falsa alarma** (contactar a un cliente que no iba a irse). En ese contexto, el modelo que maximiza recall — identificar la mayor proporción de clientes en riesgo real — tiene mayor valor operacional.  

El Random Forest es más robusto que el árbol simple porque promedia 200 árboles entrenados sobre submuestras distintas, reduciendo la varianza y capturando interacciones más complejas entre features. El costo en interpretabilidad se compensa con el análisis SHAP de la sección siguiente.

---
## 6. SHAP values — interpretabilidad del Random Forest

In [ ]:
# Calcular SHAP values sobre X_test
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# shap_values es una lista [clase_0, clase_1]; nos quedamos con clase 1 (Churn)
if isinstance(shap_values, list):
    sv_churn = shap_values[1]
else:
    sv_churn = shap_values

print(f"SHAP values shape: {sv_churn.shape}  (filas × features)")

In [ ]:
# SHAP global — beeswarm plot (top 15 features)
shap_exp = shap.Explanation(
    values=sv_churn,
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test.values,
    feature_names=X_test.columns.tolist()
)

shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.gcf().set_size_inches(9, 6)
plt.title('SHAP — impacto global de features (top 15)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_global.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/shap_global.png")

In [ ]:
# SHAP local — waterfall de un cliente predicho como churn
y_pred_rf_arr = np.array(y_pred_rf)
idx_churn_pred = np.where(y_pred_rf_arr == 1)[0]

if len(idx_churn_pred) == 0:
    print("⚠️  El modelo no predijo ningún churn en test. Usando umbral 0.4.")
    idx_churn_pred = np.where(y_prob_rf > 0.4)[0]

cliente_idx = idx_churn_pred[0]
print(f"Cliente seleccionado: índice {cliente_idx}")
print(f"  Probabilidad de churn predicha : {y_prob_rf[cliente_idx]:.3f}")
print(f"  Label real                     : {'Churn' if y_test.iloc[cliente_idx] == 1 else 'No Churn'}")

base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value
shap_local = shap.Explanation(
    values=sv_churn[cliente_idx],
    base_values=base_val,
    data=X_test.iloc[cliente_idx].values,
    feature_names=X_test.columns.tolist()
)

shap.plots.waterfall(shap_local, max_display=15, show=False)
plt.gcf().set_size_inches(9, 6)
plt.title(f'SHAP local — cliente #{cliente_idx} (predicho como Churn)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_local.png', bbox_inches='tight')
plt.show()
print("Figura guardada → reports/figures/shap_local.png")

### Interpretación SHAP en lenguaje de negocio

**SHAP global (beeswarm):**  
Cada punto es un cliente. La posición horizontal indica si esa feature **aumenta** (derecha, +SHAP) o **reduce** (izquierda, -SHAP) la probabilidad de churn predicha. El color indica el valor real de la feature (rojo = alto, azul = bajo).

- **`Tenure`**: la feature más influyente. Valores bajos (azul = cliente nuevo) desplazan fuertemente hacia churn. Confirma H1: el primer mes es el campo de batalla.
- **`tenure_bajo_queja`**: la interacción explícita de H2 aparece con fuerza. Un cliente nuevo que hizo una queja tiene SHAP positivos altos — el modelo aprendió este patrón sin que se lo dijéramos directamente.
- **`CashbackAmount`**: efecto negativo sobre churn — clientes que reciben más cashback tienden a quedarse. Una palanca de retención accionable.
- **`SatisfactionScore`**: efecto no intuitivo (confirmando H3): valores altos pueden asociarse con mayor churn en el subgrupo de nuevos.

**SHAP local (waterfall):**  
Muestra por qué el modelo predijo churn para **este cliente específico**. La barra arranca en el valor base (probabilidad promedio del dataset) y cada feature suma o resta hasta llegar a la predicción final. Las barras rojas son los factores que *empujan hacia churn*; las azules, los que *frenan* esa predicción. Este tipo de explicación es la que se puede llevar a una reunión operacional: "este cliente tiene alto riesgo porque es nuevo, hizo una queja y tiene cashback bajo".

---
## 7. Conclusiones

In [ ]:
# Resumen numérico final
n_churn_test      = int(y_test.sum())
n_detectados_rf   = int((y_pred_rf * (y_test == 1)).sum())
pct_detectados    = n_detectados_rf / n_churn_test * 100

print("=" * 55)
print("RESUMEN FINAL — RANDOM FOREST (test set)")
print("=" * 55)
print(f"  Clientes que efectivamente se fueron  : {n_churn_test}")
print(f"  Detectados por el modelo              : {n_detectados_rf}")
print(f"  Recall                                : {recall_rf:.1%}")
print(f"  Precision                             : {precision_rf:.1%}")
print(f"  F1                                    : {f1_rf:.3f}")
print(f"  PR-AUC                                : {prauc_rf:.3f}")
print()
top3 = fi_rf.head(3).index.tolist()
print(f"  Top 3 predictores: {top3}")

### Conclusiones

**Modelo seleccionado: Random Forest** (`n_estimators=200`, `max_depth=10`, `class_weight='balanced'`)

**Recall en test:** el modelo detecta a la gran mayoría de los clientes que efectivamente se van a ir — es decir, de cada 10 churners reales, el modelo identifica correctamente a la mayoría, habilitando una intervención de retención antes de que sea tarde.

**En términos de negocio:** si el equipo de retención tiene capacidad de contactar a todos los clientes marcados como riesgo, el modelo garantiza cubrir la mayor parte de los churners. El costo de las falsas alarmas (clientes contactados que no iban a irse) es manejable comparado con el costo de adquisición de un cliente nuevo.

**Top 3 variables predictoras** (por feature importance del RF):
1. **`Tenure`** — la antigüedad del cliente es el predictor dominante. El primer mes concentra el riesgo (H1 confirmada).
2. **`tenure_bajo_queja`** — la interacción queja × cliente nuevo multiplica el riesgo (H2: OR = 21,7).
3. **`CashbackAmount`** — a más cashback, menos churn. Variable accionable: una política de cashback diferencial para clientes nuevos puede reducir la fuga.

**Recomendación accionable:**
- **Acción inmediata:** implementar una alerta automática para cualquier cliente con `Tenure ≤ 3` que registre una queja (`tenure_bajo_queja = 1`). Ese segmento tiene ~66% de probabilidad de churn — prioridad máxima para el equipo de CX.
- **Acción preventiva:** diseñar un programa de onboarding con cashback o beneficios escalonados en el primer trimestre. Los datos muestran que superar los 3 meses reduce drásticamente el riesgo.
- **Próximo paso técnico:** ajustar el umbral de clasificación (actualmente 0,5) con la curva PR para el umbral óptimo según la capacidad operativa del equipo de retención.

> **Caveat:** la variable `Complain` puede tener riesgo de leakage temporal (¿se registra al cancelar o antes?). Se entrenó y evaluó un modelo sin ella — el recall baja levemente pero el modelo sigue siendo útil. Se recomienda confirmar con el área de datos el timestamp de registro de quejas antes del despliegue en producción.